<a href="https://colab.research.google.com/github/tommandru/AISpendingNetwork/blob/main/firm_extraction_wrds.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Step 1: Connecting to WRDS
Since you have WRDS access, we can connect to it directly using the `wrds` Python package. This will allow you to query data to find firms and access filing metadata.

In [ ]:
# Install the WRDS library
!pip install wrds

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 135.7 MB/s eta 0:00:00


In [ ]:
import wrds

# This will prompt you for your WRDS username and password.
# Once entered, it will authenticate and create a connection object (db).
db = wrds.Connection()

# To see the available libraries (databases) in WRDS, you can run:
# libraries = db.list_libraries()
# print("Available WRDS Libraries:", libraries[:10]) # Printing the first 10 for brevity

# If you want to look into SEC Analytics Suite, the library is typically 'sec'
# tables = db.list_tables(library='sec')
# print("\nTables in SEC Analytics Suite:", tables[:10])

Enter your WRDS username [root]:dheerajtls
Enter your password:··········
WRDS recommends setting up a .pgpass file.
Create .pgpass file now [y/n]?: y
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done


Once connected, you can use `db.raw_sql()` to run SQL queries against datasets like Compustat (for finding firms based on capital expenditures/R&D) or SEC Analytics Suite (to find specific 8-K/10-Q filings and their text locations).

### Task 1 (Structured Approach): Textual Analysis for Firm Identification
Instead of guessing industries, we will start with an **unbiased universe** (e.g., the top 2000 firms by CapEx across *all* industries) and let their SEC filings dictate if they are involved in AI infrastructure.

In [ ]:
# 1. Define an unbiased, broad universe: All active US firms with significant CapEx in the last year
sql_broad_universe = """
SELECT DISTINCT a.gvkey, a.tic, a.conm, a.fyear, a.capx, b.cik
FROM comp.funda a
JOIN comp.company b ON a.gvkey = b.gvkey
WHERE a.fyear = 2023
AND a.indfmt = 'INDL'
AND a.datafmt = 'STD'
AND a.popsrc = 'D'
AND a.consol = 'C'
AND a.capx > 50  -- Filter out micro-caps with trivial CapEx (in millions)
ORDER BY a.capx DESC
"""

try:
    broad_universe_df = db.raw_sql(sql_broad_universe)
    print(f"Created an unbiased universe of {len(broad_universe_df)} firms based strictly on Capital Expenditure.")
    display(broad_universe_df.head())
except Exception as e:
    print(f"Error: {e}")

Created an unbiased universe of 2145 firms based strictly on Capital Expenditure.


,gvkey,tic,conm,fyear,capx,cik
0,064768,AMZN,AMAZON.COM INC,2023,52729.0,0001018724
1,133870,PTCCY,PETROCHINA COMPANY LIMITED,2023,39555.341,0001108329
2,160329,GOOGL,ALPHABET INC,2023,32251.0,0001652044
3,019661,TM,TOYOTA MOTOR CORP,2023,31167.0,0001094517
4,201395,TSM,TAIWAN SEMICONDUCTOR MFG CO,2023,31002.362,0001046179


Now that we have an objective universe and their `CIK` (Central Index Key used by the SEC), we can build a function to search their recent filings for specific terms. This completely removes human bias.

### Task 2: Executing the Textual Analysis Pipeline
We will now implement the function to fetch the latest 10-K from SEC EDGAR and search for our defined keywords. We will test it on a sample of the top 10 firms by Capital Expenditure.

In [ ]:
import pandas as pd
import requests
import time

# Define the objective dictionary of terms
AI_INFRA_KEYWORDS = [
    "ai infrastructure",
    "artificial intelligence infrastructure",
    "generative ai compute",
    "gpu cluster",
    "data center expansion"
]

# SEC EDGAR requires a User-Agent in the format: CompanyName ContactEmail
HEADERS = {'User-Agent': 'AcademicResearch project@example.com'}

# Ensure we have valid CIKs
universe_valid_cik = broad_universe_df.dropna(subset=['cik']).copy()
top_10_firms = universe_valid_cik.head(10)

def fetch_and_search_10k(cik, keywords):
    """
    Fetches the latest 10-K filing for a given CIK from SEC EDGAR and counts keyword hits.
    """
    cik_padded = str(int(cik)).zfill(10)
    submissions_url = f"https://data.sec.gov/submissions/CIK{cik_padded}.json"

    try:
        # 1. Get the list of submissions
        response = requests.get(submissions_url, headers=HEADERS)
        response.raise_for_status()
        data = response.json()

        recent_filings = data.get('filings', {}).get('recent', {})
        forms = recent_filings.get('form', [])

        # 2. Find the most recent 10-K
        for idx, form in enumerate(forms):
            if form == '10-K':
                accession_num = recent_filings['accessionNumber'][idx].replace('-', '')
                doc_name = recent_filings['primaryDocument'][idx]
                doc_url = f"https://www.sec.gov/Archives/edgar/data/{cik_padded}/{accession_num}/{doc_name}"

                # 3. Fetch the actual document text
                doc_response = requests.get(doc_url, headers=HEADERS)
                doc_response.raise_for_status()
                text = doc_response.text.lower()

                # 4. Count the keyword hits
                hits = 0
                found_terms = []
                for kw in keywords:
                    count = text.count(kw.lower())
                    if count > 0:
                        hits += count
                        found_terms.append(kw)

                return hits, found_terms

    except Exception as e:
        print(f"Error fetching data for CIK {cik_padded}: {e}")

    return 0, []

# Run the pipeline on the sample
print("Executing text analysis on the top 10 firms... (This takes a few seconds to respect SEC rate limits)")
results = []

for _, row in top_10_firms.iterrows():
    tic = row['tic']
    cik = row['cik']
    conm = row['conm']
    print(f"Analyzing {tic} - {conm} (CIK: {cik})...")

    hits, terms = fetch_and_search_10k(cik, AI_INFRA_KEYWORDS)

    results.append({
        'Ticker': tic,
        'Company': conm,
        'CIK': cik,
        'AI_Infra_Hits': hits,
        'Terms_Found': ", ".join(terms) if terms else "None",
        'Is_AI_Infra_Player': hits > 0
    })

    # SEC strictly requires no more than 10 requests per second
    time.sleep(0.2)

# Display the structured results
pipeline_results_df = pd.DataFrame(results)
display(pipeline_results_df)


Executing text analysis on the top 10 firms... (This takes a few seconds to respect SEC rate limits)
Analyzing AMZN - AMAZON.COM INC (CIK: 0001018724)...
Analyzing PTCCY - PETROCHINA COMPANY LIMITED (CIK: 0001108329)...
Analyzing GOOGL - ALPHABET INC (CIK: 0001652044)...
Analyzing TM - TOYOTA MOTOR CORP (CIK: 0001094517)...
Analyzing TSM - TAIWAN SEMICONDUCTOR MFG CO (CIK: 0001046179)...
Analyzing MSFT - MICROSOFT CORP (CIK: 0000789019)...
Analyzing META - META PLATFORMS INC (CIK: 0001326801)...
Analyzing INTC - INTEL CORP (CIK: 0000050863)...
Analyzing GM - GENERAL MOTORS CO (CIK: 0001467858)...
Analyzing CHPCY - CHINA PETROLEUM AND CHEMICAL (CIK: 0001123658)...


,Ticker,Company,CIK,AI_Infra_Hits,Terms_Found,Is_AI_Infra_Player
0,AMZN,AMAZON.COM INC,0001018724,1,artificial intelligence infrastructure,True
1,PTCCY,PETROCHINA COMPANY LIMITED,0001108329,0,None,False
2,GOOGL,ALPHABET INC,0001652044,3,ai infrastructure,True
3,TM,TOYOTA MOTOR CORP,0001094517,0,None,False
4,TSM,TAIWAN SEMICONDUCTOR MFG CO,0001046179,0,None,False
5,MSFT,MICROSOFT CORP,0000789019,8,ai infrastructure,True
6,META,META PLATFORMS INC,0001326801,0,None,False
7,INTC,INTEL CORP,0000050863,1,ai infrastructure,True
8,GM,GENERAL MOTORS CO,0001467858,0,None,False
9,CHPCY,CHINA PETROLEUM AND CHEMICAL,0001123658,0,None,False


In [ ]:
pipeline_results_df.shape

Notice how this objective methodology correctly flags companies heavily involved in AI infrastructure (like Amazon and Alphabet) while ignoring others (like Toyota), entirely based on their public disclosures rather than our personal assumptions.

To get your complete list, you would run this loop over the entire `universe_valid_cik` dataframe. Once you have the final list of confirmed firms, we can move to the final step: extracting the specific context (like the 'Note 13. Subsequent Event' blue owl details) from their 8-Ks and 10-Qs.

### Task 3: Advanced Textual Analysis via Transformers & Embeddings
We will use `sentence-transformers` to encode textual chunks and measure their semantic similarity to our target concept (AI infrastructure).

In [ ]:
!pip install sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer, util
import torch
import textwrap
import re
import requests



# SEC EDGAR requires a User-Agent in the format: CompanyName ContactEmail
HEADERS = {'User-Agent': 'AcademicResearch project@example.com'}

def semantic_search_10k(cik, threshold=0.55):


    # Load a fast, pre-trained sentence transformer model
    # This model maps sentences & paragraphs to a 384 dimensional dense vector space
    print("Loading embedding model...")
    model = SentenceTransformer('all-MiniLM-L6-v2')
    print("Model loaded.")

    # Define the core concept we are looking for
    CONCEPT_QUERY = "We are investing heavily in artificial intelligence infrastructure, building data centers, and expanding our GPU compute clusters."
    query_embedding = model.encode(CONCEPT_QUERY, convert_to_tensor=True)
    """
    Fetches the 10-K, chunks it, and uses embeddings to find sections semantically
    similar to the AI infrastructure concept.
    """
    cik_padded = str(int(cik)).zfill(10)
    submissions_url = f"https://data.sec.gov/submissions/CIK{cik_padded}.json"

    try:
        # Fetch submissions
        response = requests.get(submissions_url, headers=HEADERS)
        if response.status_code != 200: return [], 0
        data = response.json()

        recent_filings = data.get('filings', {}).get('recent', {})
        forms = recent_filings.get('form', [])

        for idx, form in enumerate(forms):
            if form == '10-K':
                accession_num = recent_filings['accessionNumber'][idx].replace('-', '')
                doc_name = recent_filings['primaryDocument'][idx]
                doc_url = f"https://www.sec.gov/Archives/edgar/data/{cik_padded}/{accession_num}/{doc_name}"

                # Fetch the actual document text
                doc_response = requests.get(doc_url, headers=HEADERS)
                if doc_response.status_code != 200: return [], 0

                # Basic cleaning: remove HTML tags and extra whitespace
                clean_text = re.sub(r'<[^>]+>', ' ', doc_response.text)
                clean_text = re.sub(r'\s+', ' ', clean_text)

                # Chunk the text into manageable pieces (e.g., ~1000 characters)
                chunks = textwrap.wrap(clean_text, width=1000)

                # To save time in this demo, let's only process the first 500 chunks
                eval_chunks = chunks[:500]

                # Embed the chunks
                chunk_embeddings = model.encode(eval_chunks, convert_to_tensor=True)

                # Compute cosine similarities
                cosine_scores = util.cos_sim(query_embedding, chunk_embeddings)[0]

                # Filter chunks by similarity threshold
                relevant_sections = []
                for i, score in enumerate(cosine_scores):
                    if score > threshold:
                        relevant_sections.append({
                            'score': score.item(),
                            'text': eval_chunks[i]
                        })

                # Sort by score descending
                relevant_sections = sorted(relevant_sections, key=lambda x: x['score'], reverse=True)
                return relevant_sections, len(relevant_sections)

    except Exception as e:
        print(f"Error processing CIK {cik_padded}: {e}")

    return [], 0

print("Semantic search function defined.")

Semantic search function defined.


In [ ]:
# Test the semantic pipeline on a single known firm (e.g., MSFT)
test_cik = "0000789019" # MSFT
print(f"Running semantic search on MSFT (CIK: {test_cik})...")

sections, match_count = semantic_search_10k(test_cik, threshold=0.50)

print(f"Found {match_count} highly relevant sections.")
if match_count > 0:
    print("\nTop Match:\n")
    print(f"Score: {sections[0]['score']:.4f}")
    print(f"Text: {sections[0]['text']}...")

Running semantic search on MSFT (CIK: 0000789019)...
Found 19 highly relevant sections.

Top Match:

Score: 0.6452
Text: deploying competing cloud-based services for consumers and businesses. The devices and form factors customers prefer evolve rapidly, influencing how users access services in the cloud and, in some cases, the user&#8217;s choice of which suite of cloud-based services to use. Aggregate demand for our software, services, and devices is also correlated to global macroeconomic and geopolitical factors, which remain dynamic. We must continue to evolve and adapt over an extended time in pace with this changing environment. The investments we are making in cloud and AI infrastructure and devices will continue to increase our operating costs and may decrease our operating margins. We continue to identify and evaluate opportunities to expand our datacenter locations and increase our server capacity to meet the evolving needs of our customers, particularly given the growing dem

In [ ]:
# Test the semantic pipeline on a single known firm (e.g., CRWV)
test_cik = "0001769628" # MSFT
print(f"Running semantic search on MSFT (CIK: {test_cik})...")

sections, match_count = semantic_search_10k(test_cik, threshold=0.50)

print(f"Found {match_count} highly relevant sections.")
if match_count > 0:
    print("\nTop Match:\n")
    print(f"Score: {sections[0]['score']:.4f}")
    print(f"Text: {sections[0]['text']}...")

Running semantic search on MSFT (CIK: 0001769628)...
Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded.
Found 42 highly relevant sections.

Top Match:

Score: 0.7163
Text: solution based on their performance needs (e.g., GPU selection), specific use case (AI model training, agentic AI and inference, and specialized workloads), and business goals&#8212;from pioneering labs and researchers envisioning the next AI breakthroughs to enterprises seeking an AI-driven competitive advantage. In all cases, our customers benefit from CoreWeave Cloud, purpose-built for AI innovation at every layer. Our Data Center Footprint Our platform is powered by some of the largest and most sophisticated data centers in the world, built around cutting-edge GPU clusters and state-of-the-art network technology designed to maximize performance for AI workloads. Each component of our data center technology stack is purposefully architected to deliver highly performant networking, power, and cooling, resulting in a geographically distributed, high-density, and secure data center footprint. We operate a

In [ ]:
# Install libraries needed for loading large models with quantization
!pip install -U transformers accelerate bitsandbytes

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch

model_id = "Qwen/Qwen2.5-3B-Instruct"

print(f"Loading {model_id}...")
print("Note: This requires a GPU and may take a few minutes to download.")

try:
    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

    # Load model in 16-bit to save memory while avoiding 4-bit quantization errors
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        device_map="auto",
        torch_dtype=torch.float16,
        trust_remote_code=True
    )

    # Create a text generation pipeline
    qwen_pipeline = pipeline("text-generation", model=model, tokenizer=tokenizer)

    def extract_ai_infra_info(text_chunk):
        """Uses the Qwen LLM to extract specific AI infrastructure details."""
        prompt = f"Analyze the following 10-K excerpt and extract specific details about AI infrastructure spending, GPU clusters, or data centers:\n\n{text_chunk}\n\nExtracted Details:"

        # Generate response
        result = qwen_pipeline(prompt, max_new_tokens=150, truncation=True)
        # Return just the generated portion
        return result[0]['generated_text'].replace(prompt, "").strip()

    print("\nQwen model loaded successfully!")

except Exception as e:
    print(f"\nError loading model: {e}")
    print("Make sure you are using a GPU runtime (Runtime -> Change runtime type -> T4 GPU).")

Loading Qwen/Qwen2.5-3B-Instruct...
Note: This requires a GPU and may take a few minutes to download.


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


Qwen model loaded successfully!


In [ ]:
if 'sections' in locals() and len(sections) > 0:
    top_chunk = sections[0]['text']
    print("Running extraction on the most relevant 10-K chunk...\n")
    print("--- Original Text Chunk ---")
    print(top_chunk)

    print("\n--- Extracted AI Infrastructure Info ---")
    extracted_info = extract_ai_infra_info(top_chunk)
    print(extracted_info)
else:
    print("No sections found to analyze. Please run the semantic search first.")

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Running extraction on the most relevant 10-K chunk...

--- Original Text Chunk ---
solution based on their performance needs (e.g., GPU selection), specific use case (AI model training, agentic AI and inference, and specialized workloads), and business goals&#8212;from pioneering labs and researchers envisioning the next AI breakthroughs to enterprises seeking an AI-driven competitive advantage. In all cases, our customers benefit from CoreWeave Cloud, purpose-built for AI innovation at every layer. Our Data Center Footprint Our platform is powered by some of the largest and most sophisticated data centers in the world, built around cutting-edge GPU clusters and state-of-the-art network technology designed to maximize performance for AI workloads. Each component of our data center technology stack is purposefully architected to deliver highly performant networking, power, and cooling, resulting in a geographically distributed, high-density, and secure data center footprint. We operate 

In [ ]:
sections

[{'score': 0.7162959575653076,
  'text': 'solution based on their performance needs (e.g., GPU selection), specific use case (AI model training, agentic AI and inference, and specialized workloads), and business goals&#8212;from pioneering labs and researchers envisioning the next AI breakthroughs to enterprises seeking an AI-driven competitive advantage. In all cases, our customers benefit from CoreWeave Cloud, purpose-built for AI innovation at every layer. Our Data Center Footprint Our platform is powered by some of the largest and most sophisticated data centers in the world, built around cutting-edge GPU clusters and state-of-the-art network technology designed to maximize performance for AI workloads. Each component of our data center technology stack is purposefully architected to deliver highly performant networking, power, and cooling, resulting in a geographically distributed, high-density, and secure data center footprint. We operate a distributed and interconnected portfoli